# FLoRA Results Analysis

This notebook keeps the result analysis focused: load experiment logs, compute final-round summaries, and show Plotly charts inline. It does not export plot files.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

BASE_DIR = Path('.')
SEEDS = [0, 1, 2]
EXP_KEYS = ['tinyllama_homo', 'tinyllama_heter', 'llama_homo', 'llama_heter']

LABELS = {
    'tinyllama_homo': 'TinyLlama Homo',
    'tinyllama_heter': 'TinyLlama Heter',
    'llama_homo': 'Llama-7B Homo',
    'llama_heter': 'Llama-7B Heter',
}

MODELS = {
    'tinyllama_homo': 'TinyLlama',
    'tinyllama_heter': 'TinyLlama',
    'llama_homo': 'Llama-7B',
    'llama_heter': 'Llama-7B',
}

SETTINGS = {
    'tinyllama_homo': 'Homo',
    'tinyllama_heter': 'Heter',
    'llama_homo': 'Homo',
    'llama_heter': 'Heter',
}

COLORS = {
    'Paper': '#6C757D',
    'Replication mean': '#2A9D8F',
    'FLoRA Dirichlet mean': '#2A9D8F',
    'FLoRA stratified seed 0': '#457B9D',
    'FFA stratified seed 0': '#E76F51',
    'Linear r x 2 seed 2': '#8AB17D',
    'Nonlinear Exp2 seed 2': '#F4A261',
}


def read_score_file(path):
    path = Path(path)
    if not path.exists():
        return None
    return [float(line.strip()) * 100 for line in path.read_text().splitlines() if line.strip()]


def load_seed_scores(folder, seed, nested=False):
    if nested:
        return read_score_file(BASE_DIR / folder / f'seed{seed}' / '10' / 'log.txt')
    return read_score_file(BASE_DIR / folder / f'seed{seed}' / '10log.txt')


def load_single_scores(folder, seed=None, nested=False):
    if seed is None:
        return read_score_file(BASE_DIR / folder / '10log.txt')
    return load_seed_scores(folder, seed, nested=nested)


def load_multi_seed(folders, seeds=SEEDS, nested=False):
    return {
        key: {
            seed: scores
            for seed in seeds
            if (scores := load_seed_scores(folder, seed, nested=nested)) is not None
        }
        for key, folder in folders.items()
    }


def final_seed_stats(seed_runs):
    rows = []
    stats = {}
    for key in EXP_KEYS:
        finals = [seed_runs[key][seed][-1] for seed in SEEDS if seed in seed_runs.get(key, {})]
        if finals:
            mean = float(np.mean(finals))
            std = float(np.std(finals, ddof=1)) if len(finals) > 1 else 0.0
            min_val = float(np.min(finals))
            max_val = float(np.max(finals))
        else:
            mean = std = min_val = max_val = np.nan
        stats[key] = {'mean': mean, 'std': std, 'min': min_val, 'max': max_val, 'seeds': finals}
        rows.append({
            'Experiment': LABELS[key],
            'Seeds': ', '.join(str(seed) for seed in seed_runs.get(key, {})),
            'Final mean': mean,
            'Final std': std,
            'Min': min_val,
            'Max': max_val,
        })
    return stats, pd.DataFrame(rows)


def final_value(results, key):
    scores = results.get(key)
    return scores[-1] if scores else np.nan


def make_final_bar(df, title, y_max=None):
    fig = go.Figure()
    series = list(df['Series'].drop_duplicates())
    for name in series:
        sub = df[df['Series'] == name]
        error_array = sub['Std'] if 'Std' in sub and sub['Std'].notna().any() else None
        fig.add_bar(
            name=name,
            x=sub['Experiment'],
            y=sub['Accuracy'],
            error_y=dict(type='data', array=error_array, visible=error_array is not None),
            marker_color=COLORS.get(name),
            text=[f'{v:.2f}' if pd.notna(v) else '' for v in sub['Accuracy']],
            textposition='outside',
        )
    fig.update_layout(
        title=title,
        yaxis_title='MMLU accuracy (%)',
        barmode='group',
        template='plotly_white',
        height=520,
        legend_title_text='',
        margin=dict(t=80, r=30, b=80, l=60),
    )
    if y_max is not None:
        fig.update_yaxes(range=[0, y_max])
    fig.show()


def make_round_curve(df, title):
    fig = px.line(
        df,
        x='Round',
        y='Accuracy',
        color='Method',
        line_dash='Setting',
        facet_col='Model',
        markers=True,
        title=title,
        template='plotly_white',
        labels={'Accuracy': 'MMLU accuracy (%)'},
    )
    fig.update_layout(height=480, legend_title_text='')
    fig.update_xaxes(dtick=1)
    fig.show()


def print_loaded(title, results):
    print(title)
    for key in EXP_KEYS:
        scores = results.get(key)
        if scores is None:
            print(f'  {LABELS[key]}: missing')
        else:
            print(f'  {LABELS[key]}: {[f"{score:.2f}" for score in scores]}')

## Wizard Replication Baseline

Three training seeds are loaded for the Wizard replication baseline. The chart compares final-round means with the paper values.

In [2]:
WIZ_FOLDERS = {
    'tinyllama_homo': 'flora-tinyllama-homo-wiz',
    'tinyllama_heter': 'flora-tinyllama-heter-wiz',
    'llama_homo': 'flora-llama-homo-wiz',
    'llama_heter': 'flora-llama-heter-wiz',
}

PAPER_WIZ = {
    'tinyllama_homo': 43.87,
    'tinyllama_heter': 41.48,
    'llama_homo': 34.26,
    'llama_heter': 27.91,
}

wiz_seed_results = load_multi_seed(WIZ_FOLDERS)
wiz_stats, wiz_summary = final_seed_stats(wiz_seed_results)
wiz_summary['Paper'] = [PAPER_WIZ[key] for key in EXP_KEYS]
wiz_summary['Delta mean - paper'] = [wiz_stats[key]['mean'] - PAPER_WIZ[key] for key in EXP_KEYS]
display(wiz_summary.round(2))

wiz_bar = pd.DataFrame(
    [{'Experiment': LABELS[key], 'Series': 'Paper', 'Accuracy': PAPER_WIZ[key], 'Std': np.nan} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'Replication mean', 'Accuracy': wiz_stats[key]['mean'], 'Std': wiz_stats[key]['std']} for key in EXP_KEYS]
)
make_final_bar(wiz_bar, 'Wizard final-round MMLU: paper vs replication baseline', y_max=60)

,Experiment,Seeds,Final mean,Final std,Min,Max,Paper,Delta mean - paper
0,TinyLlama Homo,"0, 1, 2",38.63,0.41,38.28,39.08,43.87,-5.24
1,TinyLlama Heter,"0, 1, 2",41.61,0.52,41.19,42.20,41.48,0.13
2,Llama-7B Homo,"0, 1, 2",30.70,0.47,30.17,31.07,34.26,-3.56
3,Llama-7B Heter,"0, 1, 2",29.47,0.26,29.27,29.77,27.91,1.56


## Dolly Replication Baseline

This is the legacy Dolly split replication baseline with seeds 0, 1, and 2.

In [3]:
DOLLY_FOLDERS = {
    'tinyllama_homo': 'flora-tinyllama-homo-dolly',
    'tinyllama_heter': 'flora-tinyllama-heter-dolly',
    'llama_homo': 'flora-llama-homo-dolly',
    'llama_heter': 'flora-llama-heter-dolly',
}

PAPER_DOLLY = {
    'tinyllama_homo': 30.80,
    'tinyllama_heter': 18.45,
    'llama_homo': 30.99,
    'llama_heter': 28.50,
}

dolly_seed_results = load_multi_seed(DOLLY_FOLDERS)
dolly_stats, dolly_summary = final_seed_stats(dolly_seed_results)
dolly_summary['Paper'] = [PAPER_DOLLY[key] for key in EXP_KEYS]
dolly_summary['Delta mean - paper'] = [dolly_stats[key]['mean'] - PAPER_DOLLY[key] for key in EXP_KEYS]
display(dolly_summary.round(2))

dolly_bar = pd.DataFrame(
    [{'Experiment': LABELS[key], 'Series': 'Paper', 'Accuracy': PAPER_DOLLY[key], 'Std': np.nan} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'FLoRA Dirichlet mean', 'Accuracy': dolly_stats[key]['mean'], 'Std': dolly_stats[key]['std']} for key in EXP_KEYS]
)
make_final_bar(dolly_bar, 'Dolly final-round MMLU: paper vs legacy Dirichlet replication', y_max=45)

,Experiment,Seeds,Final mean,Final std,Min,Max,Paper,Delta mean - paper
0,TinyLlama Homo,"0, 1, 2",26.97,0.65,26.31,27.60,30.80,-3.83
1,TinyLlama Heter,"0, 1, 2",27.64,0.92,27.09,28.71,18.45,9.19
2,Llama-7B Homo,"0, 1, 2",32.63,0.22,32.44,32.86,30.99,1.64
3,Llama-7B Heter,"0, 1, 2",35.31,0.95,34.24,36.05,28.50,6.81


## Experiment 1: Linear Doubled Rank

Experiment 1 uses Wizard data and seed 2. It is kept as the linear expressivity comparison for the nonlinear Wizard experiment.

In [4]:
EXP1_FOLDERS = {
    'tinyllama_homo': 'exp1-tinyllama-homo-linear-r32-wiz',
    'tinyllama_heter': 'exp1-tinyllama-heter-linear-r2x-wiz',
    'llama_homo': 'exp1-llama-homo-linear-r32-wiz',
    'llama_heter': 'exp1-llama-heter-linear-r2x-wiz',
}

EXP1_SEED = 2
exp1_results = {key: load_single_scores(folder, seed=EXP1_SEED) for key, folder in EXP1_FOLDERS.items()}
print_loaded('Experiment 1 per-round scores:', exp1_results)

exp1_summary = pd.DataFrame([
    {
        'Experiment': LABELS[key],
        'Baseline mean': wiz_stats[key]['mean'],
        'Baseline std': wiz_stats[key]['std'],
        'Linear r x 2 seed 2': final_value(exp1_results, key),
        'Delta vs baseline': final_value(exp1_results, key) - wiz_stats[key]['mean'],
    }
    for key in EXP_KEYS
])
display(exp1_summary.round(2))

exp1_bar = pd.DataFrame(
    [{'Experiment': LABELS[key], 'Series': 'Replication mean', 'Accuracy': wiz_stats[key]['mean'], 'Std': wiz_stats[key]['std']} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'Linear r x 2 seed 2', 'Accuracy': final_value(exp1_results, key), 'Std': np.nan} for key in EXP_KEYS]
)
make_final_bar(exp1_bar, 'Wizard final-round MMLU: replication baseline vs linear doubled rank', y_max=60)

Experiment 1 per-round scores:
  TinyLlama Homo: ['42.23', '42.04', '43.27']
  TinyLlama Heter: ['34.00', '41.13', '43.11']
  Llama-7B Homo: ['30.04']
  Llama-7B Heter: ['26.46']


,Experiment,Baseline mean,Baseline std,Linear r x 2 seed 2,Delta vs baseline
0,TinyLlama Homo,38.63,0.41,43.27,4.65
1,TinyLlama Heter,41.61,0.52,43.11,1.50
2,Llama-7B Homo,30.70,0.47,30.04,-0.67
3,Llama-7B Heter,29.47,0.26,26.46,-3.02


## Experiment 2: Nonlinear FLoRA

Experiment 2 is Wizard-only in the current outputs, so it is analyzed separately from the Dolly stratified FFA run.

In [5]:
EXP2_FOLDERS = {
    'tinyllama_homo': 'exp2-tinyllama-nonlinear-wiz',
    'tinyllama_heter': 'exp2-tinyllama-nonlinear-3round-heter-wiz',
    'llama_homo': 'exp2-llama-nonlinear-1round-homo-wiz',
    'llama_heter': 'exp2-llama-nonlinear-1round-heter-wiz',
}

EXP2_SEED = 2
exp2_results = {key: load_single_scores(folder, seed=EXP2_SEED, nested=True) for key, folder in EXP2_FOLDERS.items()}
print_loaded('Experiment 2 per-round scores:', exp2_results)

exp2_summary = pd.DataFrame([
    {
        'Experiment': LABELS[key],
        'Baseline mean': wiz_stats[key]['mean'],
        'Baseline std': wiz_stats[key]['std'],
        'Linear r x 2 seed 2': final_value(exp1_results, key),
        'Nonlinear Exp2 seed 2': final_value(exp2_results, key),
        'NL delta vs baseline': final_value(exp2_results, key) - wiz_stats[key]['mean'],
        'NL delta vs linear r x 2': final_value(exp2_results, key) - final_value(exp1_results, key),
    }
    for key in EXP_KEYS
])
display(exp2_summary.round(2))

exp2_bar = pd.DataFrame(
    [{'Experiment': LABELS[key], 'Series': 'Replication mean', 'Accuracy': wiz_stats[key]['mean'], 'Std': wiz_stats[key]['std']} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'Linear r x 2 seed 2', 'Accuracy': final_value(exp1_results, key), 'Std': np.nan} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'Nonlinear Exp2 seed 2', 'Accuracy': final_value(exp2_results, key), 'Std': np.nan} for key in EXP_KEYS]
)
make_final_bar(exp2_bar, 'Wizard final-round MMLU: baseline vs linear doubled rank vs nonlinear FLoRA', y_max=60)

Experiment 2 per-round scores:
  TinyLlama Homo: ['36.87', '31.21', '29.60', '37.21', '26.65', '33.24']
  TinyLlama Heter: ['35.94', '28.53', '32.34', '36.74', '28.05', '32.58']
  Llama-7B Homo: ['27.25', '30.55']
  Llama-7B Heter: ['25.28', '29.49']


,Experiment,Baseline mean,Baseline std,Linear r x 2 seed 2,Nonlinear Exp2 seed 2,NL delta vs baseline,NL delta vs linear r x 2
0,TinyLlama Homo,38.63,0.41,43.27,33.24,-5.39,-10.03
1,TinyLlama Heter,41.61,0.52,43.11,32.58,-9.03,-10.53
2,Llama-7B Homo,30.70,0.47,30.04,30.55,-0.16,0.51
3,Llama-7B Heter,29.47,0.26,26.46,29.49,0.01,3.03


## Dolly Stratified FLoRA vs FFA

The FFA outputs from `run_all_seeds_ffa.sh` are Dolly stratified seed 0 runs. They are compared against the matching Dolly stratified FLoRA seed 0 outputs, with the legacy Dolly Dirichlet replication mean shown only as context.

In [6]:
DOLLY_STRATIFIED_FLORA_FOLDERS = {
    'tinyllama_homo': 'flora-tinyllama-homo-dolly_stratified',
    'tinyllama_heter': 'flora-tinyllama-heter-dolly_stratified',
    'llama_homo': 'flora-llama-homo-dolly_stratified',
    'llama_heter': 'flora-llama-heter-dolly_stratified',
}

DOLLY_STRATIFIED_FFA_FOLDERS = {
    'tinyllama_homo': 'ffa-tinyllama-homo-dolly_stratified',
    'tinyllama_heter': 'ffa-tinyllama-heter-dolly_stratified',
    'llama_homo': 'ffa-llama-homo-dolly_stratified',
    'llama_heter': 'ffa-llama-heter-dolly_stratified',
}

STRATIFIED_SEED = 0
flora_stratified_results = {
    key: load_single_scores(folder, seed=STRATIFIED_SEED)
    for key, folder in DOLLY_STRATIFIED_FLORA_FOLDERS.items()
}
ffa_stratified_results = {
    key: load_single_scores(folder, seed=STRATIFIED_SEED, nested=True)
    for key, folder in DOLLY_STRATIFIED_FFA_FOLDERS.items()
}

print_loaded('FLoRA Dolly stratified seed 0 per-round scores:', flora_stratified_results)
print()
print_loaded('FFA Dolly stratified seed 0 per-round scores:', ffa_stratified_results)

stratified_summary = pd.DataFrame([
    {
        'Experiment': LABELS[key],
        'FLoRA Dirichlet mean': dolly_stats[key]['mean'],
        'FLoRA Dirichlet std': dolly_stats[key]['std'],
        'FLoRA stratified seed 0': final_value(flora_stratified_results, key),
        'FFA stratified seed 0': final_value(ffa_stratified_results, key),
        'FFA delta vs stratified FLoRA': final_value(ffa_stratified_results, key) - final_value(flora_stratified_results, key),
        'FFA delta vs Dirichlet mean': final_value(ffa_stratified_results, key) - dolly_stats[key]['mean'],
    }
    for key in EXP_KEYS
])
display(stratified_summary.round(2))

ffa_bar = pd.DataFrame(
    [{'Experiment': LABELS[key], 'Series': 'FLoRA Dirichlet mean', 'Accuracy': dolly_stats[key]['mean'], 'Std': dolly_stats[key]['std']} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'FLoRA stratified seed 0', 'Accuracy': final_value(flora_stratified_results, key), 'Std': np.nan} for key in EXP_KEYS]
    + [{'Experiment': LABELS[key], 'Series': 'FFA stratified seed 0', 'Accuracy': final_value(ffa_stratified_results, key), 'Std': np.nan} for key in EXP_KEYS]
)
make_final_bar(ffa_bar, 'Dolly final-round MMLU: legacy baseline vs stratified FLoRA vs FFA', y_max=45)

FLoRA Dolly stratified seed 0 per-round scores:
  TinyLlama Homo: ['19.42', '28.21', '27.09']
  TinyLlama Heter: ['22.29', '28.40', '26.94']
  Llama-7B Homo: ['26.63']
  Llama-7B Heter: ['32.77']

FFA Dolly stratified seed 0 per-round scores:
  TinyLlama Homo: ['15.58', '18.03', '23.17']
  TinyLlama Heter: ['19.58', '19.49', '20.67']
  Llama-7B Homo: ['29.63']
  Llama-7B Heter: ['30.88']


,Experiment,FLoRA Dirichlet mean,FLoRA Dirichlet std,FLoRA stratified seed 0,FFA stratified seed 0,FFA delta vs stratified FLoRA,FFA delta vs Dirichlet mean
0,TinyLlama Homo,26.97,0.65,27.09,23.17,-3.93,-3.81
1,TinyLlama Heter,27.64,0.92,26.94,20.67,-6.27,-6.97
2,Llama-7B Homo,32.63,0.22,26.63,29.63,3.00,-3.00
3,Llama-7B Heter,35.31,0.95,32.77,30.88,-1.90,-4.43


In [7]:
round_rows = []
for key in EXP_KEYS:
    for method, results in [
        ('FLoRA stratified seed 0', flora_stratified_results),
        ('FFA stratified seed 0', ffa_stratified_results),
    ]:
        scores = results.get(key) or []
        for idx, score in enumerate(scores, start=1):
            round_rows.append({
                'Experiment': LABELS[key],
                'Model': MODELS[key],
                'Setting': SETTINGS[key],
                'Method': method,
                'Round': idx,
                'Accuracy': score,
            })

round_df = pd.DataFrame(round_rows)
make_round_curve(round_df, 'Dolly stratified per-round MMLU: FLoRA vs FFA')

## Controlled Epoch/Round Tuning

These cells discover `epoch_round_tuning/tuning-*` outputs from `run_epoch_round_tuning.sh`, summarize per-round validation MMLU, choose plateau pairs, and build request tables for the next sweep phase. The tuning metric is `mmlu_test_1444.jsonl`; keep `mmlu_test_14042.jsonl` for the final selected configurations.

In [8]:
from utils.tuning_analysis import (
    build_extension_requests,
    build_llama_confirmation_requests,
    build_repeat_requests,
    load_tuning_results,
    make_tuning_heatmap,
    make_tuning_round_curves,
    select_plateaus,
    summarize_tuning_results,
    write_manifest,
)

TUNING_RUN_ROOT = BASE_DIR / 'epoch_round_tuning'
tuning_scores = load_tuning_results(TUNING_RUN_ROOT)
tuning_summary = summarize_tuning_results(tuning_scores)
tuning_plateaus, tuning_selected = select_plateaus(tuning_summary)

if tuning_scores.empty:
    print(f'No tuning outputs found under {TUNING_RUN_ROOT}. Run the smoke manifest first, then the TinyLlama coarse manifest.')
else:
    print(f'Loaded {len(tuning_scores)} per-round tuning scores from {TUNING_RUN_ROOT}.')
    display(tuning_selected.round(2))
    display(tuning_plateaus.round(2))

Loaded 192 per-round tuning scores from epoch_round_tuning.


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Selected epochs,Selected round,Selected accuracy,Selected std,Selected seed count,Selected seeds,Selected compute cost,Best epochs,Best round,Best accuracy,Accuracy gap to best,Low-cost tie count
0,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,5,2,29.19,0.0,1,0,10,5,6,29.46,0.27,1
1,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,2,4,31.21,0.0,1,0,8,2,4,31.21,0.00,1
4,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,2,3,33.06,0.0,1,0,6,5,2,33.56,0.49,1
5,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,homo,Homo,1,3,31.96,0.0,1,0,3,1,6,32.59,0.63,1
2,flora,Linear FLoRA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,1,2,28.45,0.0,1,0,2,2,6,29.34,0.89,1
3,flora,Linear FLoRA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,2,2,29.62,0.0,1,0,4,2,2,29.62,0.00,1
6,flora,Linear FLoRA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,3,1,47.05,0.0,1,0,3,3,1,47.05,0.00,1
7,flora,Linear FLoRA,wiz,Wizard,tinyllama,TinyLlama,homo,Homo,1,1,47.17,0.0,1,0,1,1,1,47.17,0.00,1


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Local epochs,Plateau round,Plateau accuracy,Best round,Best accuracy,Max round observed,Declined after best
0,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,1,5,24.01,6,24.15,6,False
1,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,2,3,27.09,6,27.12,6,False
2,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,3,3,27.60,4,28.21,6,True
3,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,5,2,29.19,6,29.46,6,False
4,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,1,5,27.33,6,27.57,6,False
5,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,2,4,31.21,4,31.21,6,True
6,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,3,4,28.42,6,29.07,6,False
7,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,5,6,30.86,6,30.86,6,False
16,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,1,6,32.54,6,32.54,6,False
17,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,2,3,33.06,3,33.06,6,False


In [9]:
if not tuning_summary.empty:
    for dataset in sorted(tuning_summary['Dataset'].unique()):
        for model in sorted(tuning_summary['Model key'].unique()):
            fig = make_tuning_round_curves(tuning_summary, dataset=dataset, model=model)
            if fig is not None:
                fig.show()

if not tuning_selected.empty:
    for _, row in tuning_selected.iterrows():
        fig = make_tuning_heatmap(
            tuning_summary,
            method=row['Method key'],
            dataset=row['Dataset'],
            model=row['Model key'],
            setting=row['Setting key'],
        )
        if fig is not None:
            fig.show()

In [10]:
tuning_extension_requests = build_extension_requests(tuning_summary)
tuning_repeat_requests = build_repeat_requests(tuning_summary)
tuning_llama_requests = build_llama_confirmation_requests(tuning_selected)

for title, requests in [
    ('Extend seed-0 curves to 10 rounds', tuning_extension_requests),
    ('Repeat best TinyLlama epochs with seeds 1 and 2', tuning_repeat_requests),
    ('Llama-7B confirmation seed-0 runs', tuning_llama_requests),
]:
    print(title)
    if requests.empty:
        print('  No rows yet.')
    else:
        display(requests)

# After inspecting the tables, uncomment whichever export is needed next.
# write_manifest(tuning_extension_requests, 'tuning_manifests/tinyllama_extend.tsv')
# write_manifest(tuning_repeat_requests, 'tuning_manifests/tinyllama_repeats.tsv')
# write_manifest(tuning_llama_requests, 'tuning_manifests/llama_confirmation.tsv')

Extend seed-0 curves to 10 rounds


,method,dataset,model,setting,epochs,rounds,seed,delta_prev,delta_last
0,ffa,dolly_stratified,tinyllama,heter,5,10,0,0.546500,0.578463
1,ffa,dolly_stratified,tinyllama,homo,5,10,0,1.031187,1.595612
2,ffa,wiz,tinyllama,heter,1,10,0,0.681016,1.756371
3,ffa,wiz,tinyllama,homo,1,10,0,0.707572,1.657337
4,flora,wiz,tinyllama,homo,3,10,0,0.631018,1.417591


Repeat best TinyLlama epochs with seeds 1 and 2


,method,dataset,model,setting,epochs,rounds,seed
0,ffa,dolly_stratified,tinyllama,heter,5,6,1
1,ffa,dolly_stratified,tinyllama,heter,5,6,2
2,ffa,dolly_stratified,tinyllama,heter,3,6,1
3,ffa,dolly_stratified,tinyllama,heter,3,6,2
4,ffa,dolly_stratified,tinyllama,homo,2,6,1
5,ffa,dolly_stratified,tinyllama,homo,2,6,2
6,ffa,dolly_stratified,tinyllama,homo,5,6,1
7,ffa,dolly_stratified,tinyllama,homo,5,6,2
8,flora,dolly_stratified,tinyllama,heter,2,6,1
9,flora,dolly_stratified,tinyllama,heter,2,6,2


Llama-7B confirmation seed-0 runs


,method,dataset,model,setting,epochs,rounds,seed
0,ffa,dolly_stratified,llama,heter,1,3,0
1,ffa,dolly_stratified,llama,heter,5,3,0
2,ffa,dolly_stratified,llama,homo,1,3,0
3,ffa,dolly_stratified,llama,homo,2,3,0
4,ffa,wiz,llama,heter,1,3,0
5,ffa,wiz,llama,heter,2,3,0
6,ffa,wiz,llama,homo,1,3,0
7,flora,dolly_stratified,llama,heter,1,3,0
8,flora,dolly_stratified,llama,homo,1,3,0
9,flora,dolly_stratified,llama,homo,2,3,0


## Notes

- `run_all_sedds_ffa` is treated as `run_all_seeds_ffa.sh`.
- FFA is compared to Dolly stratified FLoRA because both use the same dataset split and seed.
- The nonlinear FLoRA Exp2 outputs currently present in this workspace are Wizard-only, so they are not mixed into the Dolly stratified FFA chart.
- Epoch/round tuning outputs are expected under `epoch_round_tuning/tuning-{method}-{dataset}-{model}-{setting}-e{epochs}-r{rounds}/seed{seed}/`.